#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

# Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

## Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df = (
  df
  .withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "M", "Married")
    .when(F.upper(F.col("cst_marital_status")) == "S", "Single")
    .otherwise("n/a")

    )
  .withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
    .when(F.upper(F.col("cst_gndr")) == "M", "Male")
    .otherwise("n/a")

    )
)


## Renaming the Colums

In [0]:
for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)

In [0]:
#trim the string
#normalization for material_status, gndr
#names are not friendly

# df.display()

# Transaction log tabele - DESCRIBE HISTORY
# spark.sql("DESCRIBE HISTORY workspace.bronze.crm_cust_info").display()

# DESCRIBE DETAIL
spark.sql("DESCRIBE DETAIL workspace.bronze.crm_cust_info").display()

# Write Into Silver Table

In [0]:
(
    df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("workspace.silver.crm_customers")
)

# HISTORY — pokazuje sve prošle operacije na tabeli
spark.sql("DESCRIBE HISTORY workspace.silver.crm_customers").display()

# DETAIL — pokazuje trenutno stanje tabele (lokacija, format, veličina...)
spark.sql("DESCRIBE DETAIL workspace.silver.crm_customers").display()

In [0]:
%sql
select * from workspace.silver.crm_customers

In [0]:
%sql
-- pogledaj staru verziju
SELECT * FROM workspace.silver.crm_customers VERSION AS OF 0


In [0]:
%sql

DESCRIBE DETAIL workspace.silver.crm_customers


In [0]:
%sql

-- ili po datumu
SELECT * FROM workspace.silver.crm_customers TIMESTAMP AS OF '2026-05-07'

In [0]:
%sql

RESTORE TABLE workspace.silver.crm_customers TO VERSION AS OF 0

In [0]:
%sql
-- dodaj property
ALTER TABLE workspace.silver.crm_customers 
SET TBLPROPERTIES ('layer' = 'silver');

-- ukloni
ALTER TABLE workspace.silver.crm_customers 
UNSET TBLPROPERTIES ('layer');

-- pogledaj
DESCRIBE DETAIL workspace.silver.crm_customers;

DESCRIBE HISTORY workspace.silver.crm_customers


In [0]:
%sql

OPTIMIZE workspace.silver.crm_customers ZORDER BY (customer_id)

In [0]:
%sql
-- briše fajlove starije od 7 dana (default)
VACUUM workspace.silver.crm_customers;

-- skrati retenciju (PAŽNJA: gubi Time Travel za taj period)
VACUUM workspace.silver.crm_customers RETAIN 24 HOURS

In [0]:
%sql
-- proveri koliko fajlova ima tvoja tabela
DESCRIBE DETAIL workspace.silver.crm_customers
-- gledaj "numFiles" kolonu

In [0]:
%sql

OPTIMIZE workspace.silver.crm_sales_details 
ZORDER BY (sales_customer_id)

In [0]:
%sql
ALTER TABLE workspace.silver.crm_customers 
SET TBLPROPERTIES ('delta.optimizeWrite' = 'true');

In [0]:
df.write \
  .mode("overwrite") \
  .option("optimizeWrite", "true") \
  .format("delta") \
  .saveAsTable("workspace.silver.crm_customers")

In [0]:
%sql
ALTER TABLE workspace.silver.crm_customers 
SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true');

In [0]:
%sql
ALTER TABLE workspace.silver.crm_customers SET TBLPROPERTIES (
  'delta.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
%sql
-- pri kreiranju tabele
CREATE TABLE workspace.silver.crm_sales_details
CLUSTER BY (sales_customer_id, sales_order_date);

-- ili na postojećoj tabeli
ALTER TABLE workspace.silver.crm_customers
CLUSTER BY (customer_id);

In [0]:
%sql
ALTER TABLE workspace.silver.crm_customers 
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');